# RF Testing

In [1]:
!pip show qickdawg

Name: qickdawg
Version: 1.2.1
Summary: Software for full quantum control of nitrogen-vacancy defects and other quantum defects in diamond
Home-page: 
Author: 
Author-email: Andy Mounce <amounce@sandia.gov>, Emmeline Riendeau <eriendeau@uchicago.edu>
License: MIT License 

Copyright 2023 National Technology & Engineering Solutions of Sandia, LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the U.S. Government retains certain rights in this software.

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substa

In [6]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
qd.start_client("128.95.31.213")

## Load Default Configuration

In [8]:
default_config = qd.NVConfiguration()

default_config.adc_channel = 0
default_config.edge_counting = True
default_config.high_threshold = 2000
default_config.low_threshold = 500


default_config.mw_channel = 0
default_config.mw_nqz = 1
default_config.mw_gain = 5000

default_config.laser_gate_pmod = 0

default_config.relax_delay_tns = 50 # between each rep, wait for everything to catch up, mostly aom


# Gain Sweep for RF Power

In [54]:
from qickdawg.testfunctions.rftest_freq import RFTestFreq
import copy

soc = qd.soc

config = copy.copy(default_config)

config.mw_channel = 0
config.mw_fMHz = 3000 # 1405 # 200
config.mw_nqz = 1

# Timing params
config.pulse_len_tus = 200
config.n_repetitions = 100000000

# Triggering (optional)
config.trigger_width_tns = 0
config.trigger_gate_pmod = 0 # Optional for triggering

# Set gain sweep (Be careful to not overload the OScilliscope or Spectrum Analyzer)
config.add_unitless_linear_sweep("gain", 0, 4000, delta=1000)

prog = RFTestFreq(config)
prog.run_rounds(
   soc,
   rounds=1, # Adjust to N cycles?
   progress=False, #True
)

config.gain_start


Requested 0 to 4000 by 1000
Instead using 0 to 5000 by 1000 in 5


0

# CPMG Super Resolution

In [10]:
from qickdawg.testfunctions.rftest_cpmg_xy_subnano_res import SUBNANO_CPMG
import copy

soc = qd.soc

config = copy.copy(default_config)

# MW params
config.mw_channel = 0
config.mw_fMHz = 20 # 1405 # 200
config.mw_gain = 5000
config.mw_nqz = 1

# Timing params
config.pi2_len_samples = int(16/3.2*40)

# Sweep params
config.n_cpmg = 32
config.reps = 1

# random
config.readout_integration_tns = 20

# Triggering
config.trigger_width_tns = 50
config.trigger_gate_pmod = 0

# tau sweep parameters (in 200ps samples)
config.add_unitless_linear_sweep("tau_samples", 500, 800, delta=2)

prog = SUBNANO_CPMG(config)
prog.run_rounds(
   soc,
   rounds=1, # Adjust to N cycles?
   progress=False, #True
)


Requested 500 to 800 by 2
Instead using 500 to 802 by 2 in 151
0b1011010


## Envelope Phase Check

In [ ]:
from qickdawg.testfunctions.rftest_envelope_phase import ENVELOPE_PHASE_CHECK
import copy

soc = qd.soc

config = copy.copy(default_config)

# MW params
config.mw_channel = 0
config.mw_fMHz = 20 # 1405 # 200
config.mw_gain = 5000
config.mw_nqz = 1

# Timing params
config.pi2_len_samples = int(16/3.2*40)

# Sweep params
config.reps = 1

# random
config.readout_integration_tns = 20

# Triggering
config.trigger_width_tns = 50
config.trigger_gate_pmod = 0

prog = ENVELOPE_PHASE_CHECK(config)
prog.run_rounds(
   soc,
   rounds=1, # Adjust to N cycles?
   progress=False, #True
)

In [69]:
import time

SiglentScope.set_single_trigger()

prog.run_rounds(
   soc,
   rounds=1, # Adjust to N cycles?
   progress=False, #True
)

time.sleep(2)
csv_name = SiglentScope.capture_scope_wvfm()

SiglentScope.plot_wvfm(csv_name)

AttributeError: 'SiglentScope' object has no attribute 'write'

In [ ]:
import time

scope = SiglentScope()              # instance = lowercase
# scope.set_single_trigger()          # set trigger

# # Run your pulse sequence / experiment
# prog.run_rounds(
#     soc,
#     rounds=1,
#     progress=False,
# )

# Wait for waveform to be acquired
time.sleep(2)

# Capture
csv_name = scope.capture_waveform()

# Plot
scope.plot_waveform(csv_name)


Saved waveform to waveform.csv


C:\Users\vsmar\AppData\Local\Temp\ipykernel_28404\1767652560.py:82: UserWarning: loadtxt: input contained no data: "waveform.csv"
  data = np.loadtxt(csv_name, delimiter=",", skiprows=1)


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [81]:
import numpy as np
import matplotlib.pyplot as plt

# Load your CSV
# Assumes CSV has two columns: Time(s), Voltage(V) with header
data = np.loadtxt(csv_file, delimiter=",", skiprows=1)

# Separate columns
time = data[:, 0]      # first column = time
voltage = data[:, 1]   # second column = voltage

# 10 MHz signal
freq_ref = config.mw_fMHz * 1e6
phi = 100/180*np.pi
ref_signals = np.array([np.sin(2 * np.pi * freq_ref * t + phi) for t in time])
ref_signals *= 3

# Plot
plt.figure(figsize=(20,4))
plt.plot(time, voltage, color='blue', linewidth=1)
plt.plot(time, ref_signals, color='red', linewidth=1, linestyle="--")
plt.title("Oscilloscope Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.grid(True)
plt.tight_layout()
plt.show()


NameError: name 'csv_file' is not defined

6 series

In [ ]:
import pyvisa, numpy as np
rm = pyvisa.ResourceManager()
scope = rm.open_resource("TCPIP0::128.95.31.184::inst0::INSTR")

# set data format
scope.write(":DATa:ENCdg ASCii")        # or RIBinary for faster transfer
scope.write(":DATa:WIDth 1")
scope.write(":DATa:STARt 1")
scope.write(":DATa:STOP 20000")         # 20 kpts

# select channel
scope.write(":DATa:SOUrce CH1")

# query scaling information
ymult = float(scope.query(":WFMOutpre:YMUlt?"))
yzero = float(scope.query(":WFMOutpre:YZEro?"))
yoff  = float(scope.query(":WFMOutpre:YOFf?"))
xincr = float(scope.query(":WFMOutpre:XINcr?"))
xzero = float(scope.query(":WFMOutpre:XZEro?"))

# read waveform
data_str = scope.query(":CURVe?")
data = np.array([float(v) for v in data_str.split(",")])
volt = (data - yoff) * ymult + yzero
time = np.arange(len(volt)) * xincr + xzero

# save to CSV
np.savetxt(f"reformatted-subnano_cpmg-{config.mw_fMHz}MHz-phase90.csv", np.column_stack([time, volt]),
           delimiter=",", header="Time(s),Voltage(V)")


In [83]:
import pyvisa
import numpy as np
import struct
import matplotlib.pyplot as plt

class SiglentScope:
    def __init__(self, ip="TCPIP0::128.95.31.184::inst0::INSTR", timeout=10000):
        self.rm = pyvisa.ResourceManager()
        self.scope = self.rm.open_resource(ip)
        self.scope.timeout = timeout
        self.scope.write_termination = '\n'
        self.scope.read_termination = '\n'

    def set_single_trigger(self, source="CH1", level=0.5, slope="POSitive"):
        """Configure the scope for a single-shot trigger."""
        self.scope.write(":TRIGger:MODE SINGle")
        self.scope.write(f":TRIGger:EDGE:SOURce {source}")
        self.scope.write(f":TRIGger:EDGE:SLOPe {slope}")
        self.scope.write(f":TRIGger:LEVel {level}")

    def wait_for_trigger(self, poll_interval=0.2):
        """Wait until the trigger fires."""
        while True:
            status = self.scope.query(":TRIGger:STATus?").strip()
            if status.upper() in ("TD", "STOP", "STOPPED", "TRIGGER"):
                break
            time.sleep(poll_interval)

    def capture_waveform(self, channel="CH1", points=40000, csv_name="waveform.csv"):
        """Capture a waveform and save it to CSV."""
        # Configure waveform
        self.scope.write(f":WAV:SOURCE {channel}")
        self.scope.write(":WAV:MODE NORM")
        self.scope.write(":WAV:FORMAT BYTE")     
        self.scope.write(f":WAV:POINTS {points}")

        # Read WAVEDESC
        self.scope.write(":WAV:PREAMBLE?")
        raw = self.scope.read_raw()

        if raw[0:1] != b'#':
            raise RuntimeError("Invalid SCPI block header")

        ndigits = int(raw[1:2])
        count = int(raw[2:2+ndigits])
        start = 2 + ndigits
        desc = raw[start:start+count]

        if len(desc) != 346:
            raise RuntimeError(f"Expected 346 bytes, got {len(desc)}")

        endian = "<"

        # Scaling fields
        (wave_array_count,) = struct.unpack_from(endian + "i", desc, 116)
        (v_gain,) = struct.unpack_from(endian + "f", desc, 156)
        (v_offset,) = struct.unpack_from(endian + "f", desc, 160)
        (h_interval,) = struct.unpack_from(endian + "f", desc, 176)
        (h_offset,) = struct.unpack_from(endian + "d", desc, 180)

        # Read waveform points
        raw_wave = self.scope.query_binary_values(":WAV:DATA?", datatype='b', container=np.array)
        raw_wave = raw_wave.astype(np.int16)

        num_points = len(raw_wave)   # <---- FIXED

        # Convert to real units
        volt = raw_wave * v_gain - v_offset
        time_axis = h_offset + np.arange(num_points) * h_interval

        # Save CSV
        np.savetxt(csv_name,
                   np.column_stack((time_axis, volt)),
                   delimiter=",",
                   header="Time(s),Voltage(V)",
                   comments="")
        print(f"Saved waveform to {csv_name}")
        return csv_name

    def plot_waveform(self, csv_name):
        """Plot a waveform from CSV."""
        data = np.loadtxt(csv_name, delimiter=",", skiprows=1)
        time = data[:, 0]
        voltage = data[:, 1]

        plt.figure(figsize=(8,4))
        plt.plot(time, voltage, linewidth=1)
        plt.title("Oscilloscope Waveform")
        plt.xlabel("Time (s)")
        plt.ylabel("Voltage (V)")
        plt.grid(True)
        plt.tight_layout()
        plt.show()
